# Pipeline Evaluation — chequeo manual post-clasificación

Objetivo: en 5 minutos verificar que las 3 tablas canónicas tienen sentido.  
Corre cada sección de arriba a abajo. Si algo se ve raro, es el lugar para anotarlo.

**Schema:** multi-mention — `reviews_classified.csv` tiene 1 fila por `(review_id, mention_id)`.  
Las métricas de negocio se calculan sobre **menciones** (sentiment, topic, urgency) o sobre **reviews únicas** (total_reviews, avg_rating).

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 60)

classified = pd.read_csv('../data/processed/reviews_classified.csv')
agg        = pd.read_csv('../data/processed/aggregated.csv')
ins        = pd.read_csv('../data/processed/insights.csv')

# cl = mention rows with successful classification
cl          = classified[classified['sentiment'].notna()].copy()
rating_only = classified[~classified['has_text']].copy()

print(f"reviews_classified : {len(classified):>4} rows  (1 per mention, not per review)")
print(f"  unique reviews   : {classified['review_id'].nunique():>4}")
print(f"  classified rows  : {len(cl)}")
print(f"  rating-only rows : {len(rating_only)}")
print(f"aggregated         : {len(agg):>4} rows")
print(f"insights           : {len(ins):>4} rows")

---
## 0 · Row vs Review — distinción multi-mention

El CSV tiene más filas que reviews porque cada review con texto genera 1–3 filas (una por topic mencionado).  
Esta sección muestra la expansión y la distribución de menciones.

In [ ]:
unique_reviews = classified['review_id'].nunique()
total_rows = len(classified)
expansion = total_rows - unique_reviews

print(f"Unique reviews : {unique_reviews}")
print(f"Total rows     : {total_rows}  (+{expansion} filas extra por expansión multi-mention)")
print()

# Mention distribution — over successfully classified text reviews
mention_dist = (
    cl.groupby('review_id')['mention_id']
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis('mentions_per_review')
    .rename('review_count')
    .to_frame()
)
mention_dist['pct'] = (mention_dist['review_count'] / mention_dist['review_count'].sum() * 100).round(1)
print("Distribución de menciones (reviews con texto clasificadas exitosamente):")
print(mention_dist.to_string())
print()
total_text = mention_dist['review_count'].sum()
multi_count = mention_dist.loc[mention_dist.index >= 2, 'review_count'].sum()
three_count = mention_dist.loc[mention_dist.index == 3, 'review_count'].sum() if 3 in mention_dist.index else 0
print(f"Reviews con ≥2 menciones : {multi_count} / {total_text}  ({multi_count/total_text*100:.1f}%)  — target: ≥30%")
print(f"Reviews con 3 menciones  : {three_count} / {total_text}  ({three_count/total_text*100:.1f}%)  — target: ≤15%")

---
## 1 · Sanidad de datos — conteos y nulos

In [ ]:
# Vista por fila (menciones) y por review única — dos lentes sobre el mismo dataset
row_view = classified.groupby('business_name').agg(
    total_rows=('review_id', 'count'),
    classified_mentions=('mention_id', lambda s: s.notna().sum()),
).astype(int)

deduped = classified.drop_duplicates(subset='review_id')
review_view = deduped.groupby('business_name').agg(
    unique_reviews=('review_id', 'count'),
    with_text=('has_text', 'sum'),
    pct_classified=('sentiment', lambda s: round(s.notna().mean() * 100, 1)),
)

pd.concat([review_view, row_view], axis=1)

In [ ]:
# Nulls en campos de clasificación (solo deberían aparecer en rating-only o clasificaciones fallidas)
cl_fields = ['mention_id', 'sentiment', 'main_topic', 'text_reference', 'urgency', 'is_actionable', 'classification_confidence']
print("Nulls per field (classified rows only):")
print(cl[cl_fields].isna().sum())
print()

# Pares (review_id, mention_id) duplicados?
dupes = classified[['review_id', 'mention_id']].dropna().duplicated().sum()
print(f"Duplicate (review_id, mention_id) pairs: {dupes}")

---
## 2 · Distribuciones — sentiment / topic / urgency / confianza

Estas métricas operan sobre **menciones** (cada fila de `cl`), no sobre reviews únicas.

In [ ]:
# Sentiment breakdown por negocio — sobre menciones
(
    cl.groupby(['business_name', 'sentiment'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
    .assign(
        pct_pos=lambda d: (d.get('positive', 0) / d['total'] * 100).round(1),
        pct_neg=lambda d: (d.get('negative', 0) / d['total'] * 100).round(1),
    )
)

In [ ]:
# Topic distribution — conteo global de menciones por topic
cl['main_topic'].value_counts()

In [ ]:
cl['urgency'].value_counts()

In [ ]:
print(cl['classification_confidence'].describe().round(3))
print(f"\nLow confidence (<0.70): {(cl['classification_confidence'] < 0.70).sum()} mentions")

---
## 3 · Spot-check — muestras aleatorias para revisión manual

Leé cada texto y verificá que el label tenga sentido. Si algo está mal anotalo más abajo.

In [ ]:
SAMPLE_N    = 20
RANDOM_SEED = 42

sample = cl.sample(SAMPLE_N, random_state=RANDOM_SEED)[
    ['business_name', 'clean_text', 'mention_id', 'sentiment', 'main_topic',
     'text_reference', 'urgency', 'is_actionable', 'classification_confidence']
].reset_index(drop=True)

sample

In [ ]:
# text_reference quality — ¿las citas son literales?
# Mismo criterio que _check_text_reference en classification/__init__.py: word-level overlap ≥ 0.8
def word_overlap(ref, text):
    words = str(ref).lower().split()
    if not words:
        return 0.0
    text_words = set(str(text).lower().split())
    return round(sum(1 for w in words if w in text_words) / len(words), 2)

sample_refs = cl[cl['text_reference'].notna()].sample(10, random_state=42)[
    ['review_id', 'clean_text', 'main_topic', 'text_reference']
].reset_index(drop=True)

sample_refs['overlap'] = sample_refs.apply(
    lambda r: word_overlap(r['text_reference'], r['clean_text']), axis=1
)

for _, r in sample_refs.iterrows():
    flag = "✓" if r['overlap'] >= 0.8 else "⚠"
    print(f"{flag} overlap={r['overlap']}  [{r['main_topic']}]")
    print(f"   ref : {r['text_reference']!r}")
    print(f"   text: {r['clean_text'][:120]!r}")
    print()

In [ ]:
# LOW CONFIDENCE — revisar estos primero (más propensos a error)
cl[cl['classification_confidence'] < 0.75][
    ['business_name', 'clean_text', 'sentiment', 'main_topic', 'urgency', 'classification_confidence']
].sort_values('classification_confidence').head(15)

In [ ]:
# HIGH URGENCY — verificar que realmente sean urgentes
cl[cl['urgency'] == 'high'][
    ['business_name', 'clean_text', 'sentiment', 'main_topic', 'text_reference', 'urgency', 'is_actionable']
]

In [ ]:
# NEGATIVE + ACTIONABLE — los más importantes para el reporte
cl[(cl['sentiment'] == 'negative') & (cl['is_actionable'] == True)][
    ['business_name', 'clean_text', 'main_topic', 'text_reference', 'urgency', 'classification_confidence']
].sort_values('urgency', ascending=False)

---
## 4 · Aggregated — chequeo de métricas de negocio

In [ ]:
agg

In [ ]:
# Verificar que pct_positive + pct_neutral + pct_negative ≈ 100 en cada fila
check = agg.copy()
check['pct_sum'] = check['pct_positive'] + check['pct_neutral'] + check['pct_negative']
print("pct sum per business (expected ~100):")
print(check[['business_name', 'pct_sum']].to_string(index=False))

---
## 5 · Insights — ranking de problemas

In [ ]:
# Top issues por negocio — estas son las recomendaciones del reporte
ins.sort_values('priority_score', ascending=False)

In [ ]:
# Por negocio — qué problema lidera en cada uno
ins.groupby('business_name').first()[['main_topic', 'priority_score', 'pct_negative', 'avg_urgency_score']]

---
## 6 · Hallazgos manuales

Usá esta celda para anotar lo que encontraste en el spot-check:

**Clasificaciones correctas:** 

**Clasificaciones dudosas o incorrectas:** 

**Patrones que el modelo no captó bien:** 

**Acciones a tomar (prompt, taxonomía, etc.):** 